In [29]:

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

from sentence_transformers import CrossEncoder
import faiss
import os


import warnings
warnings.filterwarnings("ignore")

In [30]:
#  LOAD DOCUMENTS

loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")


Loaded 2 documents


In [31]:
#  SPLIT DOCUMENTS

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

# Add metadata
for i, doc in enumerate(split_docs):
    doc.metadata["chunk_id"] = i
    doc.metadata["file_name"] = os.path.basename(doc.metadata.get("source", ""))

print(f"Split into {len(split_docs)} chunks")


Split into 18 chunks


In [32]:
#  EMBEDDINGS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1949.30it/s]


In [33]:
#  FAISS INDEX

embedding_dim = 384
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_documents(split_docs)
print("Documents indexed")


Documents indexed


In [34]:
#  LLM

llm = ChatOllama(model="llama3.2")


In [35]:
#  QUERY

query = "What is the GDP of USA in 2024?"


In [36]:
# RETRIEVAL WITH SCORES

In [37]:

docs_with_scores = vector_store.similarity_search_with_score(query, k=6)

print("\n Retrieved (Before Rerank):\n")

for i, (doc, score) in enumerate(docs_with_scores, start=1):
    print(f"{i}. Score: {score:.4f} | File: {doc.metadata.get('file_name')}")



 Retrieved (Before Rerank):

1. Score: 0.6332 | File: usa2.txt
2. Score: 0.6912 | File: usa.txt
3. Score: 0.7841 | File: usa.txt
4. Score: 0.8687 | File: usa2.txt
5. Score: 0.8944 | File: usa.txt
6. Score: 0.9346 | File: usa.txt


### retrieval scores

Lower score = more similar (because FAISS uses L2 distance)

    retriever.invoke() → only gives documents 
    
    similarity_search_with_score() → gives documents + scores

In [38]:
# RE-RANKING

In [39]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

pairs = [[query, doc.page_content] for doc, _ in docs_with_scores]

scores = reranker.predict(pairs)

ranked_docs = [
    doc for _, doc in sorted(
        zip(scores, [doc for doc, _ in docs_with_scores]),
        reverse=True
    )
]

retrieved_docs = ranked_docs[:3]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6679.54it/s]


In [40]:
retrieved_docs

[Document(id='ab0ef943-da87-4923-965f-31fe0c2f7a1c', metadata={'source': 'data\\usa.txt', 'chunk_id': 1, 'file_name': 'usa.txt'}, page_content='U.S. GDP â€“ Size, Composition, and Global Share\nAs of 2024, the United Statesâ€™ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GDP per capita is also among the highest, hovering around $83,000, which indicates a high standard of living and productivity. In terms of Purchasing Power Parity (PPP), the U.S. ranks second, but nominal GDP is considered more reflective of actual economic size and financial influence.'),
 Document(id='e8954a7a-c6f7-420a-9b32-0d530f93e38f', metadata={'source': 'data\\usa2.txt', 'chunk_id': 14, 'file_name': 'usa2.txt'}, page_content='ðŸ‡ºðŸ‡¸ U.S. GDP in 2025 â€” Economic Overview\nNominal GDP Estimate (2025)\n\nAs of 2025, the United Statesâ€™ nominal Gross Domestic

| Method       | How it works                        |
| ------------ | ----------------------------------- |
| FAISS        | compares embeddings separately      |
| CrossEncoder | reads **query + document together** |


### 🧠 Why re-ranking?

FAISS is:

    fast ✅

    but approximate ❌

CrossEncoder is:

    slow ❌

    but very accurate ✅

👉 So we combine both:
    
    FAISS → get top 6
    
    CrossEncoder → reorder them

In [41]:
#  BUILD CONTEXT 

context_parts = []

print("\n Final Selected Chunks:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    file_name = doc.metadata.get("file_name", "Unknown")
    chunk_id = doc.metadata.get("chunk_id", "N/A")

    print(f"--- Result {i} ---")
    print(f"File: {file_name}")
    print(f"Chunk ID: {chunk_id}")
    print(doc.page_content[:200], "...\n")

    context_parts.append(
        f"[Source: {file_name} | Chunk: {chunk_id}]\n{doc.page_content}"
    )

context_text = "\n\n".join(context_parts)



 Final Selected Chunks:

--- Result 1 ---
File: usa.txt
Chunk ID: 1
U.S. GDP â€“ Size, Composition, and Global Share
As of 2024, the United Statesâ€™ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ran ...

--- Result 2 ---
File: usa2.txt
Chunk ID: 14
ðŸ‡ºðŸ‡¸ U.S. GDP in 2025 â€” Economic Overview
Nominal GDP Estimate (2025)

As of 2025, the United Statesâ€™ nominal Gross Domestic Product (GDP) is estimated to be in the range of $29â€“30 trillion  ...

--- Result 3 ---
File: usa.txt
Chunk ID: 3
GDP Growth Trends and Dynamics
Historically, the U.S. economy has enjoyed consistent long-term growth, averaging around 2-3% annually. Post-pandemic, the economy bounced back strongly, but 2022 and 20 ...



doc.metadata = {
    "source": "data/economy.txt",
    "file_name": "economy.txt",
    "chunk_id": 5
}

.get() safely extracts metadata without crashing your pipeline

In [42]:
#  PROMPT

prompt = ChatPromptTemplate.from_template("""
You are a strict assistant.

Rules:
- Answer ONLY from the context
- If answer not present, say "I don't know"
- Do NOT guess

Context:
{context}

Question:
{question}

Answer:
""")

formatted_prompt = prompt.format(
    context=context_text,
    question=query
)


In [43]:
#  STREAMING RESPONSE

print("\n🤖 Answer:\n")

response_text = ""

for chunk in llm.stream(formatted_prompt):
    print(chunk.content, end="", flush=True)
    response_text += chunk.content



🤖 Answer:

$28 trillion USD

### streaming 

Instead of waiting for the full response, the LLM sends output piece by piece (token by token).
                                                                  

| Mode           | Behavior                             |
| -------------- | ------------------------------------ |
| `llm.invoke()` | waits → prints full answer           |
| `llm.stream()` | prints gradually like ChatGPT typing |


In [44]:
#  SHOW SOURCES

print("Sources Used:\n")

for doc in retrieved_docs:
    print(f"{doc.metadata.get('file_name')} | Chunk {doc.metadata.get('chunk_id')}")
    

Sources Used:

usa.txt | Chunk 1
usa2.txt | Chunk 14
usa.txt | Chunk 3


### Compare various models

| Model      | Speed    | Accuracy     | Use Case         |
| ---------- | -------- | ------------ | ---------------- |
| MiniLM-L6  | 🔥 Fast  | 👍 Good      | Default          |
| MiniLM-L12 | ⚡ Medium | 🔥 Better    | Best balance     |
| ELECTRA    | 🐢 Slow  | 🔥🔥 High    | Critical systems |
| RoBERTa    | 🐢 Slow  | 🔥🔥 High    | Deep semantics   |
| BGE-base   | ⚡ Medium | 🚀 Excellent | Modern choice    |
| BGE-large  | 🐢 Slow  | 🚀🚀 Best    | High-end         |
